In [2]:
import torch

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [4]:
!nvidia-smi

Thu Mar 20 21:53:01 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.04             Driver Version: 570.124.04     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   45C    P8              7W /   60W |      15MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [6]:
# No truncating results
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns',None)

In [7]:
# Using Practice File for faster processing, swap it to original file for correct result. 
file_path = "/home/joong/orange/ece-5464/project_3/data/k8 _practicee.xlsx"

In [8]:
df = pd.read_excel(file_path, header= None)

In [9]:
df.shape

(1900, 5409)

In [10]:
# Renaming column referred from ChatGPT 4o
df.rename(columns={df.columns[-1]: "result"}, inplace=True)

In [11]:
# Check for "?" in the entire DataFrame
question_marks = df.map(lambda x: x == "?")

# Sum up how many "?" values are in each column
question_mark_count = question_marks.sum()

print("Number of '?' values per column:\n", question_mark_count)

Number of '?' values per column:
 0         32
1         32
2         32
3         32
4         32
5         32
6         32
7         32
8         32
9         32
10        32
11        32
12        32
13        32
14        32
15        32
16        32
17        32
18        32
19        32
20        32
21        32
22        32
23        32
24        32
25        32
26        32
27        32
28        32
29        32
30        32
31        32
32        32
33        32
34        32
35        32
36        32
37        32
38        32
39        32
40        32
41        32
42        32
43        32
44        32
45        32
46        32
47        32
48        32
49        32
50        32
51        32
52        32
53        32
54        32
55        32
56        32
57        32
58        32
59        32
60        32
61        32
62        32
63        32
64        32
65        32
66        32
67        32
68        32
69        32
70        32
71        32
72        32
73        32
74  

In [12]:
# how to drop rows with all "?"
# Referred from https://saturncloud.io/blog/how-to-drop-rows-with-all-zeros-in-pandas-dataframe/

In [13]:
df.head()

0      1      2      3      4      5      6      7      8      9  \
0 -0.161 -0.014  0.002 -0.036 -0.033 -0.093  0.025  0.005      0 -0.015   
1 -0.158 -0.002 -0.012 -0.025 -0.012 -0.106  0.013  0.005      0 -0.002   
2      ?      ?      ?      ?      ?      ?      ?      ?      ?      ?   
3 -0.169 -0.025  -0.01 -0.041 -0.045 -0.069  0.038  0.014  0.008 -0.014   
4 -0.183 -0.051 -0.023 -0.077 -0.092 -0.015  0.071  0.027   0.02 -0.019   

      10     11     12     13     14     15     16     17     18     19  \
0  -0.03  -0.05 -0.031 -0.036 -0.093 -0.008  -0.03 -0.023 -0.036 -0.042   
1 -0.007  -0.01 -0.009 -0.017 -0.024  0.002  0.003 -0.011 -0.013  -0.01   
2      ?      ?      ?      ?      ?      ?      ?      ?      ?      ?   
3 -0.032 -0.043 -0.033 -0.046 -0.094 -0.017 -0.042 -0.022 -0.033 -0.039   
4 -0.044 -0.097 -0.052 -0.079 -0.175  -0.04 -0.084  -0.03 -0.047 -0.058   

      20     21     22     23     24     25     26     27     28     29  \
0 -0.038 -0.036 -0.088 -0.102 -0.021 -0.031 -0.053 -0.257 -0.114 -0.075   
1 -0.019 -0.017 -0.023 -0.044 -0.004 -0.019 -0.033 -0.187 -0.074 -0.057   
2      ?      ?      ?      ?      ?      ?      ?      ?      ?      ?   
3 -0.044  -0.04 -0.091 -0.132 -0.035 -0.036 -0.062 -0.258 -0.133 -0.097   
4 -0.067 -0.063 -0.148 -0.228 -1.008 -0.058 -0.094 -0.315 -0.204 -0.156   

      30     31     32     33     34     35     36     37     38        39  \
0 -0.037   0.08  0.027 -0.161  0.056 -0.004  0.002  0.003 -0.033 -0.056133   
1 -0.044  0.044  0.006 -0.146  0.009 -0.017 -0.009 -0.008 -0.029 -0.010433   
2      ?      ?      ?      ?      ?      ?      ?      ?      ?         ?   
3  -0.07  0.069  0.014 -0.177  0.053 -0.006 -0.002 -0.005 -0.031 -0.070167   
4 -0.122  0.081  0.007 -0.204  0.084  0.003      0 -0.008 -0.028   -0.1135   

         40     41     42     43     44     45     46     47     48     49  \
0 -0.019444  0.016  0.004  0.038  0.028 -0.016 -0.027 -0.044 -0.055 -0.024   
1  0.012889  0.023  0.022  0.026   0.06 -0.025  0.053 -0.053 -0.087 -0.029   
2         ?      ?      ?      ?      ?      ?      ?      ?      ?      ?   
3 -0.035333  0.006  -0.01  0.029 -0.013 -0.032  -0.03 -0.049 -0.084 -0.028   
4 -0.114333 -0.002 -0.032  0.034 -0.073 -0.108 -0.066 -0.073 -0.166 -0.046   

      50     51     52     53     54     55     56     57     58     59  \
0  0.016  -0.02 -0.019 -0.024  0.007 -0.046 -0.052  0.042 -0.059 -0.073   
1 -0.075 -0.122 -0.363  -0.01  0.101  0.006 -0.104 -0.029  -0.24 -0.099   
2      ?      ?      ?      ?      ?      ?      ?      ?      ?      ?   
3 -0.012 -0.051 -0.377  -0.05  0.008 -0.071 -0.073  0.019  -0.04 -0.076   
4 -0.038 -0.125 -0.423 -0.118  -0.03 -0.184 -0.133 -0.037 -0.089 -0.187   

      60     61     62     63     64     65     66     67     68     69  \
0  -0.02  0.016 -0.018 -0.023 -0.039 -0.268 -0.036  0.015 -0.003 -0.019   
1  0.027   0.12  0.086  0.199 -0.185  -0.64 -0.066  0.059  0.017  0.053   
2      ?      ?      ?      ?      ?      ?      ?      ?      ?      ?   
3  -0.04 -0.014 -0.034 -0.017 -0.043  -0.27 -0.033      0 -0.011 -0.024   
4 -0.149 -0.121  -0.12 -0.124 -0.098 -0.478 -0.068 -0.062 -0.053 -0.078   

      70     71     72     73     74     75     76     77     78     79  \
0 -0.019 -0.053 -0.037 -0.002 -0.034 -0.031 -0.034 -0.025  0.018  0.008   
1  0.084 -0.149 -0.047  0.049 -0.048 -0.012 -0.007 -0.003  0.071  0.048   
2      ?      ?      ?      ?      ?      ?      ?      ?      ?      ?   
3 -0.026  -0.05 -0.035      0 -0.023 -0.023 -0.028 -0.026  0.015  0.015   
4 -0.082 -0.121 -0.107 -0.046 -0.034  -0.04 -0.047 -0.057 -0.064  0.054   

      80     81     82     83     84     85     86     87     88     89  \
0 -0.017 -0.042 -0.035 -0.065 -0.076  0.006 -0.024 -0.023 -0.048 -0.049   
1 -0.045 -0.081 -0.029  0.093 -0.068 -0.048  0.006 -0.008 -0.012  0.006   
2      ?      ?      ?      ?      ?      ?      ?      ?      ?      ?   
3 -0.007 -0.028 -0.023 -0.069 -0.071   0.01 -0.056 -0.051  -0.0

In [14]:
# Drop rows where column 4826 has a "?"
df = df[df[4826] != "?"]

In [15]:
df.head()

0      1      2      3      4      5      6      7      8      9  \
0 -0.161 -0.014  0.002 -0.036 -0.033 -0.093  0.025  0.005      0 -0.015   
1 -0.158 -0.002 -0.012 -0.025 -0.012 -0.106  0.013  0.005      0 -0.002   
3 -0.169 -0.025  -0.01 -0.041 -0.045 -0.069  0.038  0.014  0.008 -0.014   
4 -0.183 -0.051 -0.023 -0.077 -0.092 -0.015  0.071  0.027   0.02 -0.019   
5 -0.154  0.005 -0.011 -0.013 -0.002 -0.115  0.005  0.002 -0.003  0.002   

      10     11     12     13     14     15     16     17     18     19  \
0  -0.03  -0.05 -0.031 -0.036 -0.093 -0.008  -0.03 -0.023 -0.036 -0.042   
1 -0.007  -0.01 -0.009 -0.017 -0.024  0.002  0.003 -0.011 -0.013  -0.01   
3 -0.032 -0.043 -0.033 -0.046 -0.094 -0.017 -0.042 -0.022 -0.033 -0.039   
4 -0.044 -0.097 -0.052 -0.079 -0.175  -0.04 -0.084  -0.03 -0.047 -0.058   
5  0.006 -0.002 -0.004 -0.009 -0.011  0.006  0.015 -0.008 -0.009 -0.004   

      20     21     22     23     24     25     26     27     28     29  \
0 -0.038 -0.036 -0.088 -0.102 -0.021 -0.031 -0.053 -0.257 -0.114 -0.075   
1 -0.019 -0.017 -0.023 -0.044 -0.004 -0.019 -0.033 -0.187 -0.074 -0.057   
3 -0.044  -0.04 -0.091 -0.132 -0.035 -0.036 -0.062 -0.258 -0.133 -0.097   
4 -0.067 -0.063 -0.148 -0.228 -1.008 -0.058 -0.094 -0.315 -0.204 -0.156   
5  -0.01 -0.011 -0.007 -0.016  0.005 -0.011 -0.022 -0.161 -0.058 -0.043   

      30     31     32     33     34     35     36     37     38        39  \
0 -0.037   0.08  0.027 -0.161  0.056 -0.004  0.002  0.003 -0.033 -0.056133   
1 -0.044  0.044  0.006 -0.146  0.009 -0.017 -0.009 -0.008 -0.029 -0.010433   
3  -0.07  0.069  0.014 -0.177  0.053 -0.006 -0.002 -0.005 -0.031 -0.070167   
4 -0.122  0.081  0.007 -0.204  0.084  0.003      0 -0.008 -0.028   -0.1135   
5  -0.03  0.041  0.008 -0.133      0  -0.02 -0.012  -0.01 -0.029  0.000033   

         40     41     42     43     44     45     46     47     48     49  \
0 -0.019444  0.016  0.004  0.038  0.028 -0.016 -0.027 -0.044 -0.055 -0.024   
1  0.012889  0.023  0.022  0.026   0.06 -0.025  0.053 -0.053 -0.087 -0.029   
3 -0.035333  0.006  -0.01  0.029 -0.013 -0.032  -0.03 -0.049 -0.084 -0.028   
4 -0.114333 -0.002 -0.032  0.034 -0.073 -0.108 -0.066 -0.073 -0.166 -0.046   
5  0.008278  0.018   0.01  0.043   0.05 -0.003 -0.011 -0.025 -0.014 -0.009   

      50     51     52     53     54     55     56     57     58     59  \
0  0.016  -0.02 -0.019 -0.024  0.007 -0.046 -0.052  0.042 -0.059 -0.073   
1 -0.075 -0.122 -0.363  -0.01  0.101  0.006 -0.104 -0.029  -0.24 -0.099   
3 -0.012 -0.051 -0.377  -0.05  0.008 -0.071 -0.073  0.019  -0.04 -0.076   
4 -0.038 -0.125 -0.423 -0.118  -0.03 -0.184 -0.133 -0.037 -0.089 -0.187   
5  0.008 -0.006 -0.025 -0.019 -0.006 -0.027  -0.03  0.025  -0.11 -0.114   

      60     61     62     63     64     65     66     67     68     69  \
0  -0.02  0.016 -0.018 -0.023 -0.039 -0.268 -0.036  0.015 -0.003 -0.019   
1  0.027   0.12  0.086  0.199 -0.185  -0.64 -0.066  0.059  0.017  0.053   
3  -0.04 -0.014 -0.034 -0.017 -0.043  -0.27 -0.033      0 -0.011 -0.024   
4 -0.149 -0.121  -0.12 -0.124 -0.098 -0.478 -0.068 -0.062 -0.053 -0.078   
5 -0.022  0.002 -0.021  0.017 -0.051 -0.444 -0.062  0.015 -0.027 -0.032   

      70     71     72     73     74     75     76     77     78     79  \
0 -0.019 -0.053 -0.037 -0.002 -0.034 -0.031 -0.034 -0.025  0.018  0.008   
1  0.084 -0.149 -0.047  0.049 -0.048 -0.012 -0.007 -0.003  0.071  0.048   
3 -0.026  -0.05 -0.035      0 -0.023 -0.023 -0.028 -0.026  0.015  0.015   
4 -0.082 -0.121 -0.107 -0.046 -0.034  -0.04 -0.047 -0.057 -0.064  0.054   
5  -0.03 -0.078 -0.059 -0.007 -0.064 -0.049  -0.04 -0.026  0.005 -0.009   

      80     81     82     83     84     85     86     87     88     89  \
0 -0.017 -0.042 -0.035 -0.065 -0.076  0.006 -0.024 -0.023 -0.048 -0.049   
1 -0.045 -0.081 -0.029  0.093 -0.068 -0.048  0.006 -0.008 -0.012  0.006   
3 -0.007 -0.028 -0.023 -0.069 -0.071   0.01 -0.056 -0.051  -0.05 -0.065   
4  -0.03 -0.052 -0.052 -0.184 -0.089 -0.111 -0.116 -0.087 -0.09

# impute here!!

In [16]:
from sklearn.impute import KNNImputer

# Replace '?' with np.nan explicitly
df = df.replace("?", np.nan)

/tmp/ipykernel_112661/2342245356.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("?", np.nan)


In [17]:
# Now extract X and y properly
x_df = df.drop(columns=["result"])  # Drop the target column for features
y = df["result"]  # Keep the target variable

In [18]:
# Convert any 'pd.NA' to np.nan
x_df = x_df.applymap(lambda x: np.nan if pd.isna(x) else x)

# Ensure all values are numeric
x_df = x_df.astype(float)

# n_neighbors = 1 
# Apply KNN Imputer
imputer = KNNImputer(n_neighbors=5)
x_df = imputer.fit_transform(x_df)  # Returns a NumPy array

# Convert back to DataFrame
x_df = pd.DataFrame(x_df)

# Check for remaining NaNs
print(x_df.isna().sum().sum())  # Should print 0 if all missing values are imputed

/tmp/ipykernel_112661/2703309602.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  x_df = x_df.applymap(lambda x: np.nan if pd.isna(x) else x)


0


In [19]:
print(x_df.shape, y.shape)

(1895, 5408) (1895,)


In [20]:
x_df.head(20)

0      1       2       3       4       5       6       7       8     \
0  -0.1610 -0.014  0.0020 -0.0360 -0.0330 -0.0930  0.0250  0.0050  0.0000   
1  -0.1580 -0.002 -0.0120 -0.0250 -0.0120 -0.1060  0.0130  0.0050  0.0000   
2  -0.1690 -0.025 -0.0100 -0.0410 -0.0450 -0.0690  0.0380  0.0140  0.0080   
3  -0.1830 -0.051 -0.0230 -0.0770 -0.0920 -0.0150  0.0710  0.0270  0.0200   
4  -0.1540  0.005 -0.0110 -0.0130 -0.0020 -0.1150  0.0050  0.0020 -0.0030   
5  -0.1500  0.016 -0.0140  0.0000  0.0160 -0.1230 -0.0040 -0.0020 -0.0050   
6  -0.1580  0.002 -0.0190 -0.0280 -0.0080 -0.1010  0.0110  0.0050  0.0010   
7  -0.1520  0.009 -0.0150 -0.0080  0.0040 -0.1200 -0.0020 -0.0020 -0.0070   
8  -0.1720 -0.028  0.0030 -0.0450 -0.0550 -0.0780  0.0390  0.0110  0.0060   
9  -0.1640 -0.019 -0.0110 -0.0370 -0.0310 -0.0810  0.0290  0.0120  0.0070   
10 -0.1480  0.018 -0.0120  0.0040  0.0210 -0.1280 -0.0070 -0.0030 -0.0060   
11 -0.1530  0.012 -0.0170 -0.0090  0.0060 -0.1160  0.0000  0.0010 -0.0040   
12 -0.1610 -0.014  0.0010 -0.0330 -0.0270 -0.0930  0.0230  0.0070  0.0030   
13 -0.1530  0.010 -0.0170 -0.0120  0.0040 -0.1120  0.0040  0.0020 -0.0020   
14 -0.1530  0.006 -0.0110 -0.0100  0.0000 -0.1190  0.0030  0.0000 -0.0050   
15 -0.7708 -0.012  0.0114  0.0236  0.0458 -0.1178  0.0064  0.0154  0.0104   
16 -3.2720 -0.168  0.1130  0.0640  0.0570 -0.0500  0.0840  0.0970  0.0890   
17 -0.1410  0.035 -0.0150  0.0230  0.0590 -0.1430 -0.0180 -0.0060 -0.0100   
18 -0.1660 -0.019  0.0000 -0.0410 -0.0380 -0.0930  0.0290  0.0080  0.0030   
19 -0.1750 -0.036 -0.0220 -0.0580 -0.0610 -0.0430  0.0520  0.0220  0.0150   

     9       10      11      12      13      14      15      16     17    \
0  -0.015 -0.0300 -0.0500 -0.0310 -0.0360 -0.0930 -0.0080 -0.0300 -0.023   
1  -0.002 -0.0070 -0.0100 -0.0090 -0.0170 -0.0240  0.0020  0.0030 -0.011   
2  -0.014 -0.0320 -0.0430 -0.0330 -0.0460 -0.0940 -0.0170 -0.0420 -0.022   
3  -0.019 -0.0440 -0.0970 -0.0520 -0.0790 -0.1750 -0.0400 -0.0840 -0.030   
4   0.002  0.0060 -0.0020 -0.0040 -0.0090 -0.0110  0.0060  0.0150 -0.008   
5   0.010  0.0340  0.0130  0.0090  0.0050  0.0310  0.0150  0.0370 -0.001   
6   0.003  0.0040 -0.0040 -0.0040 -0.0130 -0.0150  0.0020  0.0110 -0.007   
7   0.004  0.0220  0.0030  0.0000 -0.0030  0.0140  0.0100  0.0270 -0.007   
8  -0.020 -0.0370 -0.0830 -0.0430 -0.0530 -0.1270 -0.0220 -0.0590 -0.029   
9  -0.010 -0.0260 -0.0220 -0.0260 -0.0370 -0.0710 -0.0120 -0.0270 -0.018   
10  0.010  0.0360  0.0150  0.0100  0.0070  0.0310  0.0150  0.0390 -0.001   
11  0.008  0.0300  0.0110  0.0060  0.0000  0.0190  0.0110  0.0300 -0.003   
12 -0.013 -0.0280 -0.0360 -0.0270 -0.0330 -0.0750 -0.0080 -0.0240 -0.020   
13  0.007  0.0290  0.0090  0.0040 -0.0030  0.0080  0.0090  0.0250 -0.003   
14  0.004  0.0110  0.0000 -0.0030 -0.0070 -0.0060  0.0060  0.0180 -0.007   
15 -0.004 -0.0946 -0.2708 -0.2628  0.0248  0.0424  0.0284  0.0502  0.010   
16 -0.078 -0.6740 -1.4570 -1.3860 -0.0430  0.0020  0.0570  0.0340  0.040   
17  0.018  0.0630  0.0330  0.0250  0.0590  0.0730  0.0300  0.0680  0.007   
18 -0.017 -0.0310 -0.0540 -0.0350 -0.0400 -0.1030 -0.0190 -0.0400 -0.023   
19 -0.014 -0.0350 -0.0480 -0.0370 -0.0590 -0.1050 -0.0310 -0.0560 -0.022   

      18     19      20     21      22      23      24      25      26    \
0  -0.0360 -0.042 -0.0380 -0.036 -0.0880 -0.1020 -0.0210 -0.0310 -0.0530   
1  -0.0130 -0.010 -0.0190 -0.017 -0.0230 -0.0440 -0.0040 -0.0190 -0.0330   
2  -0.0330 -0.039 -0.0440 -0.040 -0.0910 -0.1320 -0.0350 -0.0360 -0.0620   
3  -0.0470 -0.058 -0.0670 -0.063 -0.1480 -0.2280 -1.0080 -0.0580 -0.0940   
4  -0.0090 -0.004 -0.0100 -0.011 -0.0070 -0.0160  0.0050 -0.0110 -0.0220   
5   0.0030  0.010  0.0020  0.001  0.0280  0.0230  0.0180 -0.0010 -0.0080   
6  -0.0060 -0.001 -0.0120 -0.012 -0.0050 -0.0330 -0.0020 -0.0150 -0.0280   
7  -0.0060  0.000 -0.0070 -0.007  0.0090  0.0050  0.0170 -0.0080 -0.0170   
8  -0.0480 -0.058 -0.0520 -0.049 -0.1270 -0.1530 -0.0480 -0.0410 -0.0690   
9  -0.0270 -0.026 -0.0340 -0.03

In [21]:
y.head(20)

0     inactive
1     inactive
3     inactive
4     inactive
5     inactive
6     inactive
7     inactive
8     inactive
9     inactive
10    inactive
11    inactive
12    inactive
13    inactive
14    inactive
15    inactive
16    inactive
17      active
18      active
19      active
20      active
Name: result, dtype: object

In [23]:
# This is the help I got from ChatGPT 4o to determine which rows were used to impute row 15. 
# Since n_neighbors=1 returned whatever row was closest, I had to make sure it was not just using the closest rows in my dataset. 

from sklearn.metrics.pairwise import nan_euclidean_distances
import numpy as np

# Assume x_df is your original dataframe (before imputation)
x_imputed = imputer.fit_transform(x_df)  # Imputed Data

# Compute distances to find nearest neighbors
distances = nan_euclidean_distances(x_df)
nearest_neighbors = np.argsort(distances, axis=1)[:, 1:4]  # Get top 3 nearest neighbors

# Show the 3 nearest rows used for row 15
print("Nearest neighbors for row 15:")
print(x_df.iloc[nearest_neighbors[15]])

Nearest neighbors for row 15:
     0      1      2      3      4      5      6      7      8      9     \
25 -0.148  0.024 -0.015  0.008  0.033 -0.129 -0.010 -0.003 -0.008  0.013   
16 -3.272 -0.168  0.113  0.064  0.057 -0.050  0.084  0.097  0.089 -0.078   
23 -0.142  0.039 -0.016  0.028  0.071 -0.144 -0.022 -0.009 -0.012  0.021   

     10     11     12     13     14     15     16     17     18     19    \
25  0.050  0.024  0.016  0.020  0.047  0.017  0.049  0.001  0.007  0.021   
16 -0.674 -1.457 -1.386 -0.043  0.002  0.057  0.034  0.040  0.057  0.322   
23  0.069  0.040  0.030  0.079  0.083  0.029  0.074  0.008  0.018  0.035   

     20     21     22     23     24     25     26     27     28     29    \
25  0.007  0.005  0.047  0.044  0.023  0.002  0.000 -0.096 -0.022 -0.019   
16  0.052  0.058  0.060 -0.089 -0.912  0.021  0.003 -0.274 -0.105 -0.090   
23  0.021  0.019  0.084  0.096  0.039  0.014  0.016 -0.038  0.007 -0.003   

     30     31     32     33     34     35     36     3

In [24]:
# The library was referred from https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
# Received ChatGPT 4o's help when forming nested loop

#What to fix? 
# The dataset to be trained with needs to have X and Y combined to begin with, so let's start with df 
# Then we could do imputation 

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

# Extract features and target variable
X = x_df.copy()  # Imputed feature dataset
# y = x_df_result  # Target column from df

# List of parameter values to try
weights_list = ["uniform", "distance"]
neighbors_list = [1, 3, 5, 9, 11, 15, 21]
dataset_types = ["unbalanced", "balanced"]

# To store results
results = []

# Loop over dataset types (unbalanced and balanced)
for dataset_type in dataset_types:
    if dataset_type == "unbalanced":
        X_current = X.copy()
        y_current = y.copy()
    else:
        # Create a balanced dataset using undersampling:
        balanced_df = df.copy()
        min_count = balanced_df["result"].value_counts().min()
        balanced_df = balanced_df.groupby("result", group_keys=False).apply(
            lambda grp: grp.sample(n=min_count, random_state=42)
        ).reset_index(drop=True)

        # Extract the balanced X and y
        X_current = balanced_df.drop(columns=["result"])
        y_current = balanced_df["result"]

    # Split the dataset into training and testing sets using a 70-30 split
    print(X_current.shape, y_current.shape)

    X_train, X_test, y_train, y_test = train_test_split(
        X_current, y_current, test_size=0.3, random_state=22222, stratify=y_current
    )

    # Loop over the weighting schemes and n_neighbors values
    for weight in weights_list:
        for n in neighbors_list:
            # Create the KNeighborsClassifier with the given parameters
            neigh = KNeighborsClassifier(n_neighbors=n, weights=weight)
            neigh.fit(X_train, y_train)

            # Evaluate the model
            train_score = neigh.score(X_train, y_train)
            test_score = neigh.score(X_test, y_test)

            # Store the results
            results.append({
                "dataset": dataset_type,
                "weight": weight,
                "n_neighbors": n,
                "train_score": train_score,
                "test_score": test_score
            })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)
print(results_df)


(1895, 5408) (1895,)


/tmp/ipykernel_112661/1472857962.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = balanced_df.groupby("result", group_keys=False).apply(


(38, 5408) (38,)
       dataset    weight  n_neighbors  train_score  test_score
0   unbalanced   uniform            1     1.000000    0.984183
1   unbalanced   uniform            3     0.991704    0.991213
2   unbalanced   uniform            5     0.989442    0.992970
3   unbalanced   uniform            9     0.990196    0.989455
4   unbalanced   uniform           11     0.990196    0.989455
5   unbalanced   uniform           15     0.990196    0.989455
6   unbalanced   uniform           21     0.990196    0.989455
7   unbalanced  distance            1     1.000000    0.984183
8   unbalanced  distance            3     1.000000    0.991213
9   unbalanced  distance            5     1.000000    0.994728
10  unbalanced  distance            9     1.000000    0.989455
11  unbalanced  distance           11     1.000000    0.989455
12  unbalanced  distance           15     1.000000    0.989455
13  unbalanced  distance           21     1.000000    0.989455
14    balanced   uniform            1 

### When Do I Normalize Data?? 

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")


### Scikit-Learn's kNN library explanation
##### https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html